# Goal of this notebook

In this notebook we're trying to make sense of Spark and its capabilities.

In [ ]:
from datetime import datetime, date
from pyspark.sql import Row, SparkSession 
from pyspark import SparkContext, SparkConf
import numpy as np

## Connect to Spark Cluster

Before running following block of code, you should run `run-cluster.sh` which starts local Spark cluster and Spark Connect server, so that we can connect our `SparkSession` ti the master.

In [2]:
spark = SparkSession.builder \
    .appName("chocolate sales") \
    .remote("sc://0.0.0.0:15002") \
    .getOrCreate()
    # .config("spark.jars.packages", "org.elasticsearch:elasticsearch-spark-30_2.12:8.12.0") \

# test if the cluster receives a job
df = spark.range(100)
df.write.mode("overwrite").parquet("./data/example")

Go to the web UI (http://localhost:8080) and see if the cluster received the job.

Now we can load some data.

In [3]:
df = spark.read.csv("data/calendar.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)
df.describe().show()

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

+----------+----+-----+---+----+-----------+
|      date|year|month|day|week|day_of_week|
+----------+----+-----+---+----+-----------+
|2023-01-01|2023|    1|  1|  52|          6|
|2023-01-02|2023|    1|  2|   1|          0|
|2023-01-03|2023|    1|  3|   1|          1|
|2023-01-04|2023|    1|  4|   1|          2|
|2023-01-05|2023|    1|  5|   1|          3|
+----------+----+-----+---+----+-----------+
only showing top 5 rows
+-------+------------------+-----------------+------------------+------------------+-----------------+
|summary|              year|            month|               day|              week|      day_of_week|
+-------+------------------+-----------------+------------------+------------------+-----------------+
|  count|               731|  

## Data extraction and convertion

Extract data from .h5 and store it as `pyspark.sql.DataFrame`

In [46]:
from h5extractor import *
from pyspark.sql.types import *
import pandas as pd
import itertools # for slicing a dict

In [ ]:
metadata_db, parquet_payload = process_single_h5("data/h5data.h5")
dict(itertools.islice(metadata_db.items(), 4))

{'measurement_id': 'a4725201-2621-4e3d-86ba-34424d3167e6',
 'source_file': 'data/h5data.h5'}

In [38]:
def map_type_to_spark_type(value):
    """Maps a Python/NumPy value to a PySpark DataType."""

    # Booleans (it is important to check for bool first)
    if isinstance(value, (bool, np.bool_)):
        return BooleanType()
    # Integers
    elif isinstance(value, (int, np.integer)):
        return IntegerType()
    # Floats
    elif isinstance(value, (float, np.floating)):
        return DoubleType()
    # Strings and Byte-strings (common in HDF5)
    elif isinstance(value, (str, bytes, np.bytes_, np.str_)):
        return StringType()
    # Arrays/Lists
    elif isinstance(value, (list, np.ndarray)):
        if len(value) > 0:
            # Recursively check the first element to type the array
            element_type = map_type_to_spark_type(value[0])
        else:
            # Fallback for empty arrays
            element_type = StringType() 
        return ArrayType(element_type)
    # fallback
    else:
        return StringType()

### A test

I used this to test `map_type_to_spark_type`, but for a unit test it is too slow. It can be used as a reference though

In [39]:
print("Starting debug loop...")

for key, value in metadata_db.items():
    spark_type = map_type_to_spark_type(value)
    test_schema = StructType([StructField(key, spark_type, True)])
    test_data = [{key: value}]
    
    try:
        df_test = spark.createDataFrame(data=test_data, schema=test_schema)
        df_test.collect()
        
    except Exception as e:
        print(f"\n--- FAILED ON ---")
        print(f"Key:   {key}")
        print(f"Value: {value}")
        print(f"Type:  {type(value)}")
        print(f"Mapped Spark Type: {spark_type}")
        print(f"Error: {e}")
        break

print("Debug loop finished.")

Starting debug loop...
Debug loop finished.


### Create new `pyspark.sql.dataframe`

In [44]:
fields = []

for key, value in metadata_db.items():
    spark_type = map_type_to_spark_type(value)
    field = StructField(key, spark_type, True)
    fields.append(field)

spark_schema = StructType(fields)
df = spark.createDataFrame([metadata_db], schema=spark_schema)

df.show()

+--------------------+--------------+--------------------+---------------------------------------+------------------------------------------+------------------------------------------+-----------------------------------+----------------------------------+----------------------------------+-----------------------------------+------------------------------------+-------------------------------------+--------------------------------------+----------------------------------+----------------------------------------+----------------------------------+-----------------------------------+----------------------------------------+-------------------------------+----------------------+-----------------------------+--------------------------------------+-----------------------------------------+--------------------------------------+-----------------------------------+-----------------------------------+----------------------------------+---------------------------------+--------------------------

## Elasticsearch

In [68]:
from elasticsearch import Elasticsearch, helpers

In [ ]:
def send_to_es(partition):
    # init client
    es = Elasticsearch(["http://localhost:9200"])
    
    actions = [
        {
            "_index": "chip_calibrations",
            "_source": row.asDict()
        }
        for row in partition
    ]
    
    if actions:
        helpers.bulk(es, actions)

df_metadata.rdd.foreachPartition(send_to_es)

NameError: name 'df_metadata' is not defined

In [ ]:
# df.explain(True) # very usefull before running transformations